To start: srun --partition=hpg-turin --mem=16gb --time=05:00:00 --pty bash -i
module load jupyter
launch_jupyter_notebook

In [1]:
import pandas as pd

In [13]:
!ls *.tex

ndr-30-agga-avg.tex


In [11]:
%cd /blue/anshumanc.usf/nn-infl/qwen/

/blue/anshumanc.usf/nn-infl/qwen


In [4]:
import os
from tabulate import tabulate
import re

In [7]:



def process_ndr_table(base_path: str, tasks: list[str] = ["qnli"], output_ranks = False, with_row_id = False,
                        metric_name = 30, ndr_prefix = "ndr_bl", layers = None,
                        best_group_by = None, custom_suffix = "",
                        agg_method_names = None, infl_method_names = None): 
    #NOTE: metric_name in ["f30", "auc_ndr"]):

    dfs = []
    for task in tasks:
        df = pd.read_pickle(os.path.join(base_path, f"{ndr_prefix}_{task}.pcl"))
        df.drop(columns=["scores", "noise_mask"], inplace=True)
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=False)

    if infl_method_names is not None:
        df = df.loc[df.index.get_level_values('infl').isin(infl_method_names)]
        pass

    if agg_method_names is not None:
        df = df.loc[df.index.get_level_values('agg').isin(agg_method_names)]
        pass

    if layers is not None:
        df = df.loc[df.index.get_level_values('layer').isin(layers)]
        df = df.loc[df.index.get_level_values('module') == "all"]
        pass


    metric_df = df.reset_index().pivot(index=["infl", "agg", "layer", "module"], columns=["task", "run_id"], values=[metric_name])

    ranks = metric_df.rank(axis = 0, ascending=False, method="average")
    
    suffix = ""
    if output_ranks:
        metric_df = ranks
        suffix = "-rank"

    metric_by_ds_mean = metric_df.groupby(level=1, axis=1).mean()
    metric_by_ds_std = metric_df.groupby(level=1, axis=1).std()
    metric_by_ds_std.columns = [c + "_std" for c in metric_by_ds_std.columns]
    metric_by_ds = pd.merge(metric_by_ds_mean, metric_by_ds_std, left_index=True, right_index=True)
    rank_columns = ["rank", "rank_std"]
    metric_by_ds["rank"] = ranks.mean(axis=1)
    metric_by_ds["rank_std"] = ranks.std(axis=1)

    if best_group_by is not None:
        best_idxs = metric_by_ds.groupby(level=best_group_by, axis=0)['rank'].idxmin()
        metric_by_ds = metric_by_ds.loc[best_idxs]
        metric_df = metric_df.loc[best_idxs]
        ranks = metric_df.rank(axis = 0, ascending=False, method="average")
        metric_by_ds["rank"] = ranks.mean(axis=1)
        metric_by_ds["rank_std"] = ranks.std(axis=1)
    
    metric_by_ds = metric_by_ds.sort_values(by=['rank', 'rank_std'], ascending=[True, True])

    metric_by_ds = metric_by_ds[[*[de for d in tasks for de in [d, d + "_std"]], *rank_columns]]
    # metric_by_ds.to_csv(f"{base_path}/{metric_name}{suffix}-avg.csv")

    values_to_highlight = metric_by_ds[tasks].to_numpy().max(axis=0)

    rows = []
    for row_id, row in enumerate(metric_by_ds.reset_index().to_dict(orient="records")):
        new_row = {}
        infl_method = row["infl"]
        agg_method = row["agg"]
        layer = row["layer"]
        module = row["module"].replace("embed_tokens", "WE")
        if with_row_id:
            new_row["id"] = (row_id + 1)
        new_row["infl"] = infl_method
        new_row["agg"] = agg_method
        find_layer = re.match(r"layers\s(\d+)\s", module)
        if find_layer is not None:
            layer = find_layer.group(1)
            module = module.replace(find_layer.group(0), "")
        new_row["layer"] = "WE" if module == "WE" else layer
        new_row["module"] = module
        for did, d in enumerate([*tasks, "rank"]):
            should_highlight = False
            if d == "rank":
                m = round(row[d] * 10) / 10
                m_std = round(row[d + "_std"] * 10) / 10
            else:
                should_highlight = row[d] == values_to_highlight[did]
                m = round(row[d] * 1000) / 10
                m_std = round(row[d + "_std"] * 1000) / 10
            m = str(m).rstrip("0").rstrip(".").lstrip("0").replace("-0.", "-.")
            m_std = str(m_std).rstrip("0").rstrip(".").lstrip("0")

            if should_highlight:
                if m_std == "":
                    new_row[d] = f"\\textbf{{{m}}}"
                else:
                    new_row[d] = f"\\textbf{{{m}}} $\pm$ {m_std}"
            else:
                if m_std == "":
                    new_row[d] = m
                else:
                    new_row[d] = f"{m} $\pm$ {m_std}"
        rows.append(new_row)

    with open(f"{base_path}/ndr-{metric_name}{suffix}{custom_suffix}-avg.tex", "w") as stats_file:
        s = tabulate(rows, headers = "keys", showindex=False, tablefmt="latex")
        s = s.replace("\\textbackslash{}", "\\").replace("\\$", "$").replace("hf\_we\_topk\_10", "hf$^{10}_{we}$").replace("hf\_we\_", "hf$_{we}$").replace("\\_", "_").replace("\{", "{").replace("\}", "}").replace("\^{}", "^").replace("lllllllllll", "ll|ccccccccc").replace("rand", "\\hline rand")\
            .replace("_10", "$^{10}$").replace("_50", "$^{50}$")\
            .replace("commonset", "cset").replace("-20", "$^{20}$").replace("-30", "$^{30}$")\
            .replace("commonsubset", "csset").replace("out_proj", "proj")\
            .replace('self_attn ', '')\
            .replace('v_proj', 'value').replace('q_proj', 'query')
        print(s, file = stats_file)

    pass

In [12]:
dss = ["mrpc", "qnli", "sst2", "qqp", "cola", "mnli", "rte", "stsb"]

process_ndr_table("./", tasks=dss, with_row_id=False, custom_suffix = "-agga", 
                    best_group_by=["infl", "agg"], 
                #   layers=selected_layers, 
                    ndr_prefix="ndr_agga",
                #   agg_method_names=agg_method_names
                    )

/scratch/local/18603686/ipykernel_3528672/1480102731.py:37: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  metric_by_ds_mean = metric_df.groupby(level=1, axis=1).mean()
/scratch/local/18603686/ipykernel_3528672/1480102731.py:38: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  metric_by_ds_std = metric_df.groupby(level=1, axis=1).std()
/scratch/local/18603686/ipykernel_3528672/1480102731.py:46: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  best_idxs = metric_by_ds.groupby(level=best_group_by, axis=0)['rank'].idxmin()
